# Non-Adaptive Conformal Experiments

Includes non-adaptive methods compared in the paper (Split Residual with/without FWER 
and with/without tolerance, split-conformal weighted/unweighted, clustered conformal) and runs them on
synthetic IID data and four distribution-shift regimes.

In [ ]:
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import binom
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from xgboost import XGBRegressor

In [ ]:
def weighted_quantile(arr, q, weights):
    """Weighted quantile of arr at level q; returns inf if level unreachable."""
    arr = arr.flatten()
    idx = np.argsort(arr, axis=None)
    arr = arr[idx]
    weights = weights[idx]
    weights = np.cumsum(weights)
    weights /= weights[-1] + 1
    if np.sum(weights < q) >= len(arr):
        return np.inf
    return float(arr[np.sum(weights < q)].item())

def calc_coverage_low_upp(test_y, lower, upper):
    test_y = np.asarray(test_y).reshape(-1)
    lower = np.asarray(lower).reshape(-1)
    upper = np.asarray(upper).reshape(-1)
    return float(np.mean((test_y >= lower) & (test_y <= upper)))

def exp_decay_weights(n, rho):
    """Length-n weight vector with geometric decay (oldest = rho ** (n-1))."""
    if rho == 1:
        return np.ones(n)
    w = np.ones(n)
    for i in range(1, n):
        w[i] = w[i - 1] * rho
    return w[::-1]

In [ ]:
def _predict_full(stage_1, stage_2, data, stage_1_features, stage_2_features):
    """Two-stage prediction y_hat = stage_2(stage_1(x), aux)."""
    s1_in = data[0:stage_1_features, :].reshape(-1, stage_1_features)
    s1_pred = stage_1.predict(s1_in)
    if stage_2_features > 1:
        aux = data[stage_1_features + 1 : stage_1_features + stage_2_features, :].T
        return stage_2.predict(np.concatenate((s1_pred, aux), axis=1))
    return stage_2.predict(s1_pred)

def simple_trad_resid(
    conformal_data, test_data, stage_1, stage_2,
    stage_1_features, stage_2_features, alpha=0.05, rho=0.99,
):
    """Split-conformal baseline using only the end-to-end residual.

    Returns intervals for both the exponentially decaying weighted quantile
    (label \"trad_weighted\") and the unweighted quantile (\"trad_unweighted\").
    """
    full_predicted = _predict_full(
        stage_1, stage_2, conformal_data, stage_1_features, stage_2_features,
    )
    traditional_residual = np.abs(conformal_data[-1, :] - full_predicted)

    ws = exp_decay_weights(traditional_residual.shape[0], rho)
    w = weighted_quantile(traditional_residual, 1 - alpha, ws)
    if w == np.inf:
        return -np.inf, np.inf, 0, np.inf, 0, np.inf, np.inf, np.inf, np.inf

    w_unweighted = np.quantile(
        traditional_residual, 1 - alpha, method="closest_observation"
    )
    full_test_pred = _predict_full(
        stage_1, stage_2, test_data, stage_1_features, stage_2_features,
    )
    full_test_upper, full_test_lower = w + full_test_pred, full_test_pred - w
    full_test_upper_unw = w_unweighted + full_test_pred
    full_test_lower_unw = full_test_pred - w_unweighted

    cov = calc_coverage_low_upp(test_data[-1, :].T, full_test_lower, full_test_upper)
    cov_unw = calc_coverage_low_upp(
        test_data[-1, :].T, full_test_lower_unw, full_test_upper_unw
    )
    return (
        full_test_lower, full_test_upper, full_test_pred, w, cov,
        full_test_lower_unw, full_test_upper_unw, w_unweighted, cov_unw,
    )

def cluster_residual(
    conformal_data, test_data, stage_1, stage_2,
    stage_1_features, stage_2_features, k=5, alpha=0.05, beta=1,
):
    """Clustered cascaded confidence conformal baseline (Gong et al., 2023)."""
    if beta is None:
        beta = 1
    full_predicted = _predict_full(
        stage_1, stage_2, conformal_data, stage_1_features, stage_2_features,
    )
    stage_2_predict = stage_2.predict(
        conformal_data[stage_1_features : stage_1_features + stage_2_features, :]
        .reshape(-1, stage_2_features)
    )
    r2 = np.abs(conformal_data[-1, :] - stage_2_predict)
    r1 = np.abs(full_predicted - stage_2_predict)

    # Clusters are unused for the final width below (kept for parity with the
    # original implementation, which returns global quantiles).
    n1 = int(min(max(k, 1), conformal_data.shape[1]))
    n2 = int(min(max(k, 1), conformal_data.shape[1]))
    KMeans(n_clusters=n1, random_state=0, n_init=10).fit(
        conformal_data[0:stage_1_features, :].T
    )
    KMeans(n_clusters=n2, random_state=0, n_init=10).fit(
        conformal_data[stage_1_features : stage_1_features + stage_2_features, :].T
    )

    r1_level = np.clip(1 - alpha, 0, 1)
    r2_level = np.clip(1 - beta + (1 - alpha), 0, 1)
    comb_width = np.quantile(r1, r1_level) + np.quantile(r2, r2_level)

    full_test_pred = _predict_full(
        stage_1, stage_2, test_data, stage_1_features, stage_2_features,
    )
    full_test_upper = comb_width + full_test_pred
    full_test_lower = full_test_pred - comb_width
    cov = calc_coverage_low_upp(test_data[-1, :].T, full_test_lower, full_test_upper)
    return full_test_lower, full_test_upper, full_test_pred, comb_width, cov

In [ ]:
def fixed_sequence_testing(lambda_set, pvals, delta, initializations, pvals_other=()):
    """Multiple hypothesis testing accept set over the lambda grid."""
    lambda_val = []
    cutoff = delta / len(initializations)
    for j in initializations:
        if pvals[j] > cutoff:
            continue
        if pvals_other and pvals_other[j] > cutoff:
            continue
        lambda_val.append(tuple(lambda_set[j, :]))
    return lambda_val

def fwer_interval(
    calibration_data, test_data, lambda_set, stage_1, stage_2,
    stage_1_features, stage_2_features, train_data, alpha, delta,
    conformal_data, initializations=None, eps1=0.1, eps2=0.1,
    tol1=0.00, tol2=0.00,
):
    """FWER-validated split-conformal interval over a lambda grid.

    Computes per-stage residual quantiles r1, r2 on `conformal_data`, then
    accepts (a, b) values whose calibration coverage passes a FWER test
    against alpha+tol1 (and optionally alpha-tol2).
    Among accepted (a, b), picks the one whose calibration coverage is
    closest to 1 - alpha.
    """
    conf_predicted = _predict_full(
        stage_1, stage_2, conformal_data, stage_1_features, stage_2_features,
    )
    stage_2_predict = stage_2.predict(
        conformal_data[stage_1_features : stage_1_features + stage_2_features, :]
        .reshape(-1, stage_2_features)
    )
    traditional_residual = np.abs(conformal_data[-1, :] - conf_predicted)
    r2 = np.abs(conformal_data[-1, :] - stage_2_predict)
    r1 = np.abs(traditional_residual - r2)
    r2_train = np.quantile(r2, 1 - eps1)
    r1_train = np.quantile(r1, 1 - eps2)

    full_predicted = _predict_full(
        stage_1, stage_2, calibration_data, stage_1_features, stage_2_features,
    )
    num_cal = calibration_data.shape[1]

    pvals, pvals_other_side = [], []
    for a, b in lambda_set:
        width = r1_train * a + r2_train * b
        empirical_fail = num_cal * (
            1 - calc_coverage_low_upp(
                calibration_data[-1, :].reshape(-1, 1),
                full_predicted - width, full_predicted + width,
            )
        )
        pvals.append(binom.cdf(empirical_fail, num_cal, alpha + tol1))
        if tol2 != 0:
            pvals_other_side.append(
                1 - binom.cdf(empirical_fail, num_cal, alpha - tol2)
            )

    if initializations is None:
        initializations = np.arange(len(lambda_set))

    lambda_val = fixed_sequence_testing(
        lambda_set, pvals, delta, initializations, pvals_other_side
    )
    if not lambda_val:
        return -np.inf, np.inf, 0, np.inf, np.inf, np.inf, 0, np.inf, np.inf, []

    a, b, best_cov = 0, 0, 0
    for al, bl in lambda_val:
        comb_width_l = al * r1_train + bl * r2_train
        cov_l = calc_coverage_low_upp(
            conformal_data[-1, :],
            conf_predicted - comb_width_l, conf_predicted + comb_width_l,
        )
        if np.abs(cov_l - (1 - alpha)) < np.abs(best_cov - (1 - alpha)):
            best_cov, a, b = cov_l, al, bl

    cal_width = r1_train * a + r2_train * b
    full_test_pred = _predict_full(
        stage_1, stage_2, test_data, stage_1_features, stage_2_features,
    )
    full_test_upper = cal_width + full_test_pred
    full_test_lower = full_test_pred - cal_width
    cov = calc_coverage_low_upp(test_data[-1, :].T, full_test_lower, full_test_upper)
    return (
        full_test_lower, full_test_upper, full_test_pred, cal_width,
        r1_train, r2_train, cov, a, b, lambda_val,
    )

In [ ]:
def _split_odd_even(x):
    return x[::2], x[1::2]

def _plot_method(ax, means_w, stds_w, means, stds, colors, alphas, handles):
    for i, (m1, s1, m2, s2) in enumerate(zip(means_w, stds_w, means, stds)):
        c = colors[i]
        h = ax.plot(m1, m2, "o", color=c)[0]
        ax.plot([m1 - s1, m1 + s1], [m2, m2], color=c)
        ax.plot([m1, m1], [m2 - s2, m2 + s2], color=c)
        if i == len(alphas) - 2:
            handles.append(h)

def plot_cov_width_figure(
    sr_mean, sr_meanw, notol_mean, notol_meanw,
    sc_mean, sc_meanw, wsc_mean, wsc_meanw,
    cc_mean=None, cc_meanw=None,
    out_path="placeholder.png",
    alphas=(0.3, 0.2, 0.15, 0.13, 0.1),
):
    """Coverage-vs-width scatter for the non-adaptive methods.

    Each *_mean / *_meanw is a list of [mean, std] pairs across the alpha
    grid (one pair per alpha). Point opacity within a series encodes 1-alpha.
    """
    alphas = list(alphas)
    fig, ax = plt.subplots(figsize=(7, 7))
    handles = []
    labels = ["SR", "SR-no-Tol", "SC", "WSC", "CC"]

    series = [
        (sr_mean, sr_meanw, "Blues"),
        (notol_mean, notol_meanw, "Oranges"),
        (sc_mean, sc_meanw, "Greens"),
        (wsc_mean, wsc_meanw, "Purples"),
    ]
    #Optionally skip plotting CC
    if cc_mean is not None or cc_meanw is not None:
        series.append((cc_mean, cc_meanw, "Greys"))
    for stat, statw, cmap_name in series:
        means, stds = _split_odd_even(np.asarray(stat).flatten())
        meansw, stdsw = _split_odd_even(np.asarray(statw).flatten())
        colors = plt.get_cmap(cmap_name)(np.linspace(0.2, 1, len(alphas)))
        _plot_method(ax, meansw, stdsw, means, stds, colors, alphas, handles)

    ax.legend(handles, labels, title="Methods")
    smc = plt.cm.ScalarMappable(
        norm=plt.Normalize(vmin=min(1 - a for a in alphas), vmax=max(1 - a for a in alphas)),
        cmap=plt.get_cmap("Blues"),
    )
    smc.set_array([])
    fig.colorbar(smc, ax=ax).set_label("α (Darker = Smaller α)")
    ax.set_xlabel("Interval Width")
    ax.set_ylabel("Empirical Coverage")
    plt.savefig(out_path, dpi=300)
    plt.show()

In [ ]:
METHODS = ("fwer_a", "fwer_b", "trad_unweighted", "trad_weighted", "cluster")

def compute_methods(
    conformal, calibration, conformal_isolated, train, test,
    stage_1, stage_2, stage_1_features, stage_2_features,
    alpha, delta, lambda_set, epsilon,
    fwer_kwargs_a=None, fwer_kwargs_b=None,
    rho=0.99, cluster_beta=0.9,
):
    """Run the five methods on one (calibration, test) split.

    Returns a dict METHOD -> (coverage, width). FWER is called twice with
    different tolerance kwargs to produce 'fwer_a' (e.g. with slack) and
    'fwer_b' (e.g. without).
    """
    base = dict(eps1=epsilon, eps2=epsilon)
    ka = {**base, **(fwer_kwargs_a or {})}
    kb = {**base, **(fwer_kwargs_b or {})}
    args = (
        calibration, test, lambda_set, stage_1, stage_2,
        stage_1_features, stage_2_features, train, alpha, delta,
        conformal_isolated,
    )
    out_a = fwer_interval(*args, **ka)
    out_b = fwer_interval(*args, **kb)
    out_t = simple_trad_resid(
        conformal, test, stage_1, stage_2,
        stage_1_features, stage_2_features, alpha, rho=rho,
    )
    out_c = cluster_residual(
        conformal, test, stage_1, stage_2,
        stage_1_features, stage_2_features, alpha=alpha, beta=cluster_beta,
    )
    return {
        "fwer_a":          (out_a[6], out_a[3]),
        "fwer_b":          (out_b[6], out_b[3]),
        "trad_unweighted": (out_t[8], out_t[7]),
        "trad_weighted":   (out_t[4], out_t[3]),
        "cluster":         (out_c[4], float(np.mean(out_c[3]))),
    }

In [ ]:
def save_stats(stats, path, extra_rows=()):
    """Save stats dict to .npy in the legacy 10-row layout used by the
    original notebook (with optional extra rows appended, e.g. mistake tables).
    """
    rows = []
    for m in METHODS:
        rows.append(np.array(stats[m]["cov"]).flatten())
        rows.append(np.array(stats[m]["width"]).flatten())
    for r in extra_rows:
        rows.append(np.array(r).flatten())
    mat = np.vstack(rows)
    np.save(path, mat)
    return mat

def load_stats(path):
    """Inverse of save_stats; the trailing mistake rows are ignored."""
    mat = np.load(path)
    stats = {}
    for i, m in enumerate(METHODS):
        stats[m] = {
            "cov": mat[2 * i].reshape(-1, 2).tolist(),
            "width": mat[2 * i + 1].reshape(-1, 2).tolist(),
        }
    return stats

def plot_stats(stats, out_path):
    plot_cov_width_figure(
        stats["fwer_a"]["cov"],          stats["fwer_a"]["width"],
        stats["fwer_b"]["cov"],          stats["fwer_b"]["width"],
        stats["trad_unweighted"]["cov"], stats["trad_unweighted"]["width"],
        stats["trad_weighted"]["cov"],   stats["trad_weighted"]["width"],
        stats["cluster"]["cov"],         stats["cluster"]["width"],
        out_path=out_path,
    )

In [ ]:
def run_iid_experiment(
    sample_data, fit_models,
    stage_1_features, stage_2_features,
    lambda_set, epsilon, delta,
    fwer_kwargs_a=None, fwer_kwargs_b=None,
    n_reps=50, alphas=(0.3, 0.2, 0.15, 0.13, 0.1),
    train_size=3000, conformal_size=1000, test_size=1000,
    rho=0.99, cluster_beta=0.9,
):
    """Average all five methods over n_reps samples for each alpha.

    `sample_data()` returns the full (features+response, T) matrix; the loop
    slices into train / conformal / calibration / conformal_isolated / test data

    Can be used to sweep over parameters such as delta or tol by for looping
    and fixing alphas to a singleton in a list
    """
    stats = {m: {"cov": [], "width": []} for m in METHODS}
    for alpha in alphas:
        bucket = {m: {"cov": [], "width": []} for m in METHODS}
        for _ in range(n_reps):
            full_data = sample_data()
            train = full_data[:, :train_size]
            conformal = full_data[:, train_size : train_size + conformal_size]
            calibration = conformal[:, : conformal_size // 2]
            conformal_isolated = conformal[:, conformal_size // 2 :]
            test = full_data[:, -test_size:]
            stage_1, stage_2 = fit_models(train)
            out = compute_methods(
                conformal, calibration, conformal_isolated, train, test,
                stage_1, stage_2, stage_1_features, stage_2_features,
                alpha, delta, lambda_set, epsilon,
                fwer_kwargs_a, fwer_kwargs_b, rho, cluster_beta,
            )
            for m in METHODS:
                bucket[m]["cov"].append(out[m][0])
                bucket[m]["width"].append(out[m][1])
        for m in METHODS:
            stats[m]["cov"].append([np.mean(bucket[m]["cov"]), np.std(bucket[m]["cov"])])
            stats[m]["width"].append([np.mean(bucket[m]["width"]), np.std(bucket[m]["width"])])
    return stats

In [ ]:
def run_shift_experiment(
    sample_data, fit_models,
    stage_1_features, stage_2_features,
    lambda_set, epsilon, delta,
    fwer_kwargs_a=None, fwer_kwargs_b=None,
    n_reps=50, alphas=(0.3, 0.2, 0.15, 0.13, 0.1),
    train_amount=500, size=1000, window_before_shift=100,
    rho=0.999, cluster_beta=0.9,
):
    """Sliding-window experiment under a distribution shift.

    `sample_data()` returns (full_data, shift_location). The inner loop steps
    a single test point at a time through the last `shift_location +
    window_before_shift` indices, growing the conformal set.
    Coverage / width are averaged across the window;
    if FWER returns inf for the rep, that rep is dropped and its abstention rate
    is recorded (because this occurs in shift) in the trailing two (labeled mistake) .npy rows.
    """
    stats = {m: {"cov": [], "width": []} for m in METHODS}
    mistake_a_stat, mistake_b_stat = [], []
    for alpha in alphas:
        bucket = {m: {"cov": [], "width": []} for m in METHODS}
        mistake_a, mistake_b = [], []
        for _ in range(n_reps):
            full_data, shift_location = sample_data()
            train = full_data[:, :train_amount]
            stage_1, stage_2 = fit_models(train)
            denom = shift_location + window_before_shift
            accums = {m: {"cov": 0.0, "width": 0.0} for m in METHODS}
            misses_a = misses_b = 0
            n_dim = full_data.shape[0]
            for t in range(size - shift_location - window_before_shift, size):
                conformal = full_data[:, train_amount:t]
                calibration = conformal[:, ::1]
                conformal_isolated = conformal[:, ::2]
                test = full_data[:, t].reshape(n_dim, 1)
                out = compute_methods(
                    conformal, calibration, conformal_isolated, train, test,
                    stage_1, stage_2, stage_1_features, stage_2_features,
                    alpha, delta, lambda_set, epsilon,
                    fwer_kwargs_a, fwer_kwargs_b, rho, cluster_beta,
                )
                for m in METHODS:
                    accums[m]["cov"] += out[m][0]
                    accums[m]["width"] += out[m][1]
                if out["fwer_a"][1] == np.inf:
                    misses_a += 1
                if out["fwer_b"][1] == np.inf:
                    misses_b += 1
            for m in METHODS:
                accums[m]["cov"] /= denom
                accums[m]["width"] /= denom

            # Drop rep if FWER diverged (matches original semantics).
            if accums["fwer_a"]["width"] == np.inf:
                mistake_a.append(misses_a / denom)
                continue
            mistake_a.append(0)
            if accums["fwer_b"]["width"] == np.inf:
                mistake_b.append(misses_b / denom)
                continue
            mistake_b.append(0)
            for m in METHODS:
                bucket[m]["cov"].append(accums[m]["cov"])
                bucket[m]["width"].append(accums[m]["width"])
        for m in METHODS:
            stats[m]["cov"].append([np.mean(bucket[m]["cov"]), np.std(bucket[m]["cov"])])
            stats[m]["width"].append([np.mean(bucket[m]["width"]), np.std(bucket[m]["width"])])
        mistake_a_stat.append([np.mean(mistake_a), np.std(mistake_a)])
        mistake_b_stat.append([np.mean(mistake_b), np.std(mistake_b)])
    return stats, mistake_a_stat, mistake_b_stat

In [ ]:
def make_lambda_grid(low, high, n):
    grid = np.linspace(low, high, n)
    la, lb = np.meshgrid(grid, grid)
    return np.column_stack((la.flatten(), lb.flatten()))

def fit_ols_pair(train, reg=0.1):
    """Two ridge regressions: w -> x and x -> y."""
    ols1 = sm.OLS(train.T[:, 1], train.T[:, 0]).fit_regularized(alpha=reg, L1_wt=0)
    ols2 = sm.OLS(train.T[:, 2], train.T[:, 1]).fit_regularized(alpha=reg, L1_wt=0)
    return ols1, ols2

## 1. Synthetic IID, linear

`w → x → y` with Gaussian noise at each stage. 

In [ ]:
np.random.seed(42)

def sample_iid_linear(size=5000):
    w = np.random.normal(0, 0.1, size)
    x = w * 3 + np.random.normal(0, 0.1, size)
    y = x * 4 + np.random.normal(0, 0.1, size)
    return np.vstack([w, x, y])

stats = run_iid_experiment(
    sample_data=sample_iid_linear,
    fit_models=fit_ols_pair,
    stage_1_features=1, stage_2_features=1,
    lambda_set=make_lambda_grid(0.25, 1, 10),
    epsilon=0.06, delta=0.4,
    fwer_kwargs_a={"tol1": 0.05},
    fwer_kwargs_b={"tol1": 0.00},
)
save_stats(stats, "iid")

In [ ]:
plot_stats(load_stats("iid.npy"), "empirical_cov_width_tol_iid.png")

## 2. Synthetic IID, nonlinear

Synthetic health-style data: `(age, bmi, smoker) → disease (logistic) → cost (XGB)`.

In [ ]:
np.random.seed(43)

def sample_iid_nonlinear(size=5000):
    age = np.random.randint(20, 80, size=size)
    bmi = np.random.uniform(18, 40, size=size)
    smoker = np.random.binomial(1, 0.3, size=size)
    logits = -5 + 0.05 * age + 0.1 * bmi + 1.5 * smoker
    disease = np.random.binomial(1, 1 / (1 + np.exp(-logits)))
    cost = 0.5 + 0.02 * age + 0.05 * bmi + disease * 3 + np.random.normal(0, 1, size=size)
    return np.vstack([np.vstack([age, bmi, smoker]), disease, cost])

def fit_logreg_xgb(train):
    clf = LogisticRegression().fit(train[:3, :].T, train[3, :])
    xgb = XGBRegressor(
        n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42,
    )
    xgb.fit(train[3, :].reshape(-1, 1), train[4, :])
    return clf, xgb

stats = run_iid_experiment(
    sample_data=sample_iid_nonlinear,
    fit_models=fit_logreg_xgb,
    stage_1_features=3, stage_2_features=1,
    lambda_set=make_lambda_grid(0.3, 1, 10),
    epsilon=0.06, delta=0.1,
    fwer_kwargs_a={"tol1": 0.05},
    fwer_kwargs_b={"tol1": 0.00},
)
save_stats(stats, "nonlinear")

## 3. Non-IID phi-mixing

AR(1) latent `w`, deterministic transitions to `x` and `y`. 

In [ ]:
np.random.seed(44)

def sample_phi_mixing(T=5000, phi=0.5):
    eps = np.random.uniform(-1, 1, size=T)
    eta = np.random.uniform(-0.1, 0.1, size=T)
    xi = np.random.uniform(-0.1, 0.1, size=T)
    w = np.zeros(T); x = np.zeros(T); y = np.zeros(T)
    x[0] = 3 * w[0] + eta[0]
    y[0] = 4 * y[0] + xi[0]
    for t in range(1, T):
        w[t] = phi * w[t - 1] + eps[t]
        x[t] = 3 * w[t] + eta[t]
        y[t] = 4 * x[t] + xi[t]
    return np.vstack([w, x, y])

stats = run_iid_experiment(
    sample_data=sample_phi_mixing,
    fit_models=fit_ols_pair,
    stage_1_features=1, stage_2_features=1,
    lambda_set=make_lambda_grid(0.25, 1, 10),
    epsilon=0.01, delta=0.5,
)
save_stats(stats, "phimat")

## 4. Gradual downstream shift

Drift in the `x → y` relationship over the last `shift_location` steps.

In [ ]:
np.random.seed(45)

def sample_grad_downstream(size=1000, shift_location=200):
    w = np.random.normal(0, 0.1, size)
    x = w * 3 + np.random.normal(0, 0.1, size)
    y = x * 4 + np.random.normal(0, 0.1, size)
    for i in range(size - shift_location, size):
        drift = (i - size + shift_location) / 300
        y[i] = x[i] * (4 + drift) + np.random.normal(drift, 0.5)
    return np.vstack([w, x, y]), shift_location

stats, mistake_a, mistake_b = run_shift_experiment(
    sample_data=sample_grad_downstream,
    fit_models=fit_ols_pair,
    stage_1_features=1, stage_2_features=1,
    lambda_set=make_lambda_grid(0.25, 1, 10),
    epsilon=0.02, delta=0.1,
    fwer_kwargs_a={"tol1": 0.05},
    fwer_kwargs_b={"tol1": 0.00},
)
save_stats(stats, "gradual_downstream", extra_rows=(mistake_a, mistake_b))

In [ ]:
plot_stats(load_stats("gradual_downstream.npy"), "empirical_cov_width_tol_grad_down.png")

## 5. Gradual upstream shift

Drift in the `w → x` relationship over the last `shift_location` steps.

In [ ]:
np.random.seed(46)

def sample_grad_upstream(size=1000, shift_location=200):
    w = np.random.normal(0, 0.1, size)
    x = w * 3 + np.random.normal(0, 0.1, size)
    y = x * 4 + np.random.normal(0, 0.1, size)
    for i in range(size - shift_location, size):
        drift = (i - size + shift_location) / 300
        x[i] = w[i] * (3 - drift) + np.random.normal(drift, 0.5)
    y = x * 4 + np.random.normal(0, 0.1, size)
    return np.vstack([w, x, y]), shift_location

stats, mistake_a, mistake_b = run_shift_experiment(
    sample_data=sample_grad_upstream,
    fit_models=fit_ols_pair,
    stage_1_features=1, stage_2_features=1,
    lambda_set=make_lambda_grid(0.1, 1, 8),
    epsilon=0.01, delta=0.3,
    fwer_kwargs_a={"tol1": 0.05},
    fwer_kwargs_b={"tol1": 0.00},
)
save_stats(stats, "gradual_upstream", extra_rows=(mistake_a, mistake_b))

In [ ]:
plot_stats(load_stats("gradual_upstream.npy"), "empirical_cov_width_tol_grad_up.png")

## 6. Rapid downstream shift

Sudden, random regime change in `x → y` at `shift_location`.

In [ ]:
np.random.seed(47)

def sample_rapid_downstream(size=1000, shift_location=200):
    w = np.random.normal(0, 0.1, size)
    x = w * 3 + np.random.normal(0, 0.1, size)
    y = x * 4 + np.random.normal(0, 0.1, size)
    for i in range(size - shift_location, size):
        y[i] = x[i] * (4 + np.random.normal(0, 2)) + np.random.normal(0, 1)
    return np.vstack([w, x, y]), shift_location

stats, mistake_a, mistake_b = run_shift_experiment(
    sample_data=sample_rapid_downstream,
    fit_models=fit_ols_pair,
    stage_1_features=1, stage_2_features=1,
    lambda_set=make_lambda_grid(0.1, 1, 8),
    epsilon=0.01, delta=0.1,
    fwer_kwargs_a={"tol1": 0.05},
    fwer_kwargs_b={"tol2": 0.00},
)
save_stats(stats, "rapid_downstream", extra_rows=(mistake_a, mistake_b))

In [ ]:
plot_stats(load_stats("rapid_downstream.npy"), "empirical_cov_width_tol_rap_down.png")

## 7. Rapid upstream shift

Sudden, random regime change in `w → x` at `shift_location`. 

In [ ]:
np.random.seed(48)

def sample_rapid_upstream(size=1000, shift_location=200):
    w = np.random.normal(0, 0.1, size)
    x = w * 3 + np.random.normal(0, 0.1, size)
    y = x * 4 + np.random.normal(0, 0.1, size)
    for i in range(size - shift_location, size):
        x[i] = w[i] * (3 - np.random.normal(0, 1)) + np.random.normal(0, 1)
    y = x * 4 + np.random.normal(0, 0.1, size)
    return np.vstack([w, x, y]), shift_location

stats, mistake_a, mistake_b = run_shift_experiment(
    sample_data=sample_rapid_upstream,
    fit_models=fit_ols_pair,
    stage_1_features=1, stage_2_features=1,
    lambda_set=make_lambda_grid(0.2, 1, 8),
    epsilon=0.01, delta=0.3,
    fwer_kwargs_a={"tol1": 0.05},
    fwer_kwargs_b={"tol1": 0.00},
)
save_stats(stats, "rapid_upstream", extra_rows=(mistake_a, mistake_b))

In [ ]:
plot_stats(load_stats("rapid_upstream.npy"), "empirical_cov_width_tol_rapid_up.png")